In [4]:
!pip install torch-geometric


In [28]:
!pip install stim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 67.4 MB/s eta 0:00:00


In [1]:
import stim

In [2]:
import torch
import torch.nn.functional as F



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.sta

In [3]:
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data, Batch
from torch_geometric.utils import add_self_loops, degree


In [4]:
import matplotlib.pyplot as plt


In [5]:
import numpy as np
from tqdm.auto import trange
import networkx as nx

c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# ============================================================================
# PART 1: SURFACE CODE CIRCUIT
# ============================================================================

def surface_code_circuit(p, d):
    """Generate surface code circuit with given error rate and distance"""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p
    )

In [9]:
# ============================================================================
# PART 2: SYNDROME GRAPH - Pre-built structure, only values change
# ============================================================================

class SyndromeGraph:
    """
    Pre-built graph structure for a surface code.
    The skeleton (edges) is fixed. Only node values change per sample.
    """

    def __init__(self, circuit, device):
        self.device = device
        self.num_nodes = circuit.num_detectors

        # Build the fixed graph skeleton ONCE
        self.edge_index = self._build_edges(circuit).to(device)
        self.num_edges = self.edge_index.shape[1]

        print(f"✓ Graph skeleton built: {self.num_nodes} detectors, {self.num_edges} connections")
        print(f"  Average neighbors per detector: {self.num_edges / self.num_nodes:.1f}")

    def _build_edges(self, circuit):
        """Build edge connections based on detector positions"""
        detector_coords = circuit.get_detector_coordinates()
        edges = []

        # Connect detectors that are physically adjacent
        for i in range(self.num_nodes):
            coord_i = detector_coords.get(i, [0, 0, 0])
            for j in range(i + 1, self.num_nodes):
                coord_j = detector_coords.get(j, [0, 0, 0])

                # Distance in 2D space
                dist = ((coord_i[0] - coord_j[0])**2 +
                        (coord_i[1] - coord_j[1])**2) ** 0.5

                # Neighbors are within distance 2
                if dist < 2.0:
                    edges.append([i, j])
                    edges.append([j, i])  # Bidirectional

        if len(edges) == 0:
            print("  Warning: No spatial edges found, using fallback chain")
            for i in range(self.num_nodes - 1):
                edges.append([i, i + 1])
                edges.append([i + 1, i])

        return torch.tensor(edges, dtype=torch.long).t().contiguous()

    def create_batch(self, syndrome_values):
        """
        Create batched graph data from syndrome measurements.
        EFFICIENT: Structure is pre-built, just plugs in new values.

        Args:
            syndrome_values: Tensor (batch_size, num_nodes) with +1/-1 values

        Returns:
            PyG Data object ready for GNN
        """
        batch_size = syndrome_values.shape[0]

        # Stack all node features: (batch_size * num_nodes, 1)
        x = syndrome_values[:, :self.num_nodes].reshape(-1, 1).float()

        # Replicate edges for each graph in batch (with offsets)
        edge_indices = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            edge_indices.append(self.edge_index + offset)

        edge_index = torch.cat(edge_indices, dim=1)

        # Batch assignment: which graph each node belongs to
        batch = torch.arange(batch_size, device=self.device).repeat_interleave(self.num_nodes)

        return Data(x=x, edge_index=edge_index, batch=batch)

In [10]:
# ============================================================================
# PART 4: SIMPLE GCN LAYER (NO COMPILED DEPENDENCIES)
# ============================================================================

class SimpleGCNConv(MessagePassing):
    """Graph Convolutional Layer - works without pyg-lib"""

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.bias = torch.nn.Parameter(torch.Tensor(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # Add self-loops to adjacency
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Transform features
        x = self.lin(x)

        # Compute normalization (symmetric normalization)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Message passing
        out = self.propagate(edge_index, x=x, norm=norm)

        return out + self.bias

    def message(self, x_j, norm):
        # Normalize messages from neighbors
        return norm.view(-1, 1) * x_j

In [12]:
# ============================================================================
# PART 4: GNN DECODER MODEL
# ============================================================================

class SyndromeDecoder(torch.nn.Module):
    """
    GNN that takes syndrome graphs and predicts logical error.
    Each detector shares info with neighbors through message passing.
    """

    def __init__(self, hidden_dim=128, num_layers=4):
        super().__init__()

        # Message passing layers
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # First layer: 1 input (syndrome value) -> hidden
        self.layers.append(SimpleGCNConv(1, hidden_dim))
        self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(SimpleGCNConv(hidden_dim, hidden_dim))
            self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Output: aggregate graph -> predict error
        self.output = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Message passing: nodes share info with neighbors
        for layer, norm in zip(self.layers, self.norms):
            x = layer(x, edge_index)
            x = norm(x)
            x = F.silu(x)

        # Pool: aggregate all nodes in each graph
        num_graphs = int(batch.max().item()) + 1
        graph_features = torch.zeros(num_graphs, x.size(1), device=x.device)
        for i in range(num_graphs):
            mask = (batch == i)
            graph_features[i] = x[mask].mean(dim=0)

        # Predict logical error
        return self.output(graph_features)

In [ ]:
# ============================================================================
# PART 5: GNN TRAINER - Clean interface with progress tracking
# ============================================================================

class GNNTrainer:
    """Clean interface for training GNN decoder with clear progress tracking"""

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing GNN Trainer for d={d}, p={p}")
        print(f"{'='*60}")

        # Build circuit (ONCE)
        print("\n[1/3] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build graph structure (ONCE)
        print("\n[2/3] Building graph skeleton...")
        self.graph = SyndromeGraph(self.circuit, device)

        # Build model
        print("\n[3/3] Building neural network...")
        self.model = SyndromeDecoder(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        self.loss_fn = torch.nn.BCELoss()

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"✓ Model built: {num_params:,} parameters")
        print(f"\n{'='*60}")
        print("Ready to train!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use .copy() to ensure contiguous array, then torch.from_numpy() to avoid dtype inference issues
        syndrome_np = (detections.astype(np.int64) * 2 - 1).copy()
        labels_np = flips.astype(np.int64).flatten().copy()

        syndromes = torch.from_numpy(syndrome_np).to(self.device)
        labels = torch.from_numpy(labels_np).to(self.device)

        return syndromes, labels

    def train_epoch(self, syndromes, labels, batch_size=256):
        """One training epoch with progress bar"""
        self.model.train()
        num_samples = syndromes.shape[0]
        num_batches = num_samples // batch_size

        total_loss = 0
        correct = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = start + batch_size

                batch_syn = syndromes[start:end]
                batch_lab = labels[start:end]

                # Create graph batch (structure pre-built, just plug in values)
                batch_data = self.graph.create_batch(batch_syn)

                # Forward
                pred = self.model(batch_data)
                loss = self.loss_fn(pred, batch_lab.unsqueeze(1).float())

                # Backward
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Track metrics
                total_loss += loss.item()
                batch_correct = ((pred > 0.5).flatten() == batch_lab).sum().item()
                correct += batch_correct

                # Update progress bar
                running_acc = correct / ((batch_idx + 1) * batch_size)
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{running_acc:.4f}'
                })

        avg_loss = total_loss / num_batches
        accuracy = correct / (num_batches * batch_size)
        return avg_loss, accuracy

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_samples)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                pred = self.model(batch_data)
                all_preds.append((pred > 0.5).flatten())

        all_preds = torch.cat(all_preds)
        accuracy = (all_preds == labels).float().mean().item()
        return accuracy

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_shots)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_shots, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                pred = self.model(batch_data)
                all_preds.append((pred > 0.5).flatten().cpu().numpy())

        all_preds = np.concatenate(all_preds)
        errors = (all_preds != labels.cpu().numpy())
        return errors.mean()

    def save(self, path):
        """Save model weights"""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights"""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [ ]:
    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use torch.from_numpy() to avoid dtype inference issues
        syndromes = torch.from_numpy((detections.astype(np.int64) * 2 - 1)).to(self.device)
        labels = torch.from_numpy(flips.astype(np.int64).flatten()).to(self.device)

        return syndromes, labels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/626.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 624.6/626.1 kB 22.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.1/626.1 kB 16.7 MB/s eta 0:00:00


In [14]:
# ============================================================================
# PART 6: MWPM BASELINE FOR COMPARISON
# ============================================================================

def get_mwpm_accuracy(p, d, num_shots=100000):
    """Compute MWPM accuracy for comparison"""
    import pymatching

    print(f"Computing MWPM baseline ({num_shots:,} shots)...")

    circuit = surface_code_circuit(p, d)
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = sum(
        1 for shot in range(num_shots)
        if not np.array_equal(observable_flips[shot], predictions[shot])
    )

    ler = num_errors / num_shots
    accuracy = 1 - ler
    print(f"✓ MWPM Accuracy: {accuracy:.6f} (LER: {ler:.6f})")

    return accuracy

In [15]:
# ============================================================================
# PART 7: PROGRESSIVE TRAINING - Train until beating MWPM
# ============================================================================

def train_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train with progressively larger datasets until beating MWPM"""
    import gc

    print(f"\n{'#'*60}")
    print(f"# PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline first
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up GNN")
    print("="*60)
    trainer = GNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 100
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model for fresh start
        trainer.model = SyndromeDecoder(hidden_dim=128, num_layers=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=1e-3)

        # Calculate chunks needed
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")
            print(f"  Generating data...")

            syndromes, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs per chunk for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(syndromes, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Train Acc={acc:.4f}")

            # Cleanup
            del syndromes, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     GNN Accuracy:  {gnn_accuracy:.6f}")
        print(f"     MWPM Accuracy: {mwpm_accuracy:.6f}")
        print(f"     Difference:    {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [ ]:
# ============================================================================
# PART 8: RUN THE EXPERIMENT
# ============================================================================

import os

# Configuration
train_p = 0.005
max_train_size = 10**8
chunk_size = 10**7

results = {}
trained_models = {}

# Train for each distance
for d in [3, 5, 7]:
    model_path = f"gnn_decoder_d{d}_p{train_p}.pt"

    print(f"\n{'█'*60}")
    print(f"█  DISTANCE {d}")
    print(f"{'█'*60}")

    # Check if model already exists
    if os.path.exists(model_path):
        print(f"\n✓ Found existing model: {model_path}")
        trainer = GNNTrainer(train_p, d, device)
        trainer.load(model_path)
        train_size = None
    else:
        print(f"\n→ No saved model found. Training new model...")
        trainer, train_size = train_until_beat_mwpm(
            p=train_p, d=d, device=device,
            max_train_size=max_train_size, chunk_size=chunk_size
        )

        if trainer is not None:
            trainer.save(model_path)
        else:
            print(f"Training failed for d={d}")
            continue

    trained_models[d] = trainer
    results[d] = train_size

# Print summary
print(f"\n\n{'═'*60}")
print("FINAL RESULTS SUMMARY")
print(f"{'═'*60}")
for d, train_size in results.items():
    if train_size:
        print(f"  Distance {d}: Beat MWPM with {train_size:,} samples")
    else:
        print(f"  Distance {d}: Loaded from saved model")
print(f"{'═'*60}")


Training new model for d=3...

Progressive Training for d=3, p=0.005
MWPM accuracy to beat: 0.982810
Graph structure: 24 nodes, 56 edges

Trying train_size = 100
  Chunk 1/1: 100 samples


  0%|          | 0/1 [00:00<?, ?it/s]

GNN accuracy: 0.100600 vs MWPM: 0.982810

Trying train_size = 1,000
  Chunk 1/1: 1,000 samples


  0%|          | 0/3 [00:00<?, ?it/s]

GNN accuracy: 0.104400 vs MWPM: 0.982810

Trying train_size = 10,000
  Chunk 1/1: 10,000 samples


...

  0%|          | 0/3906 [00:00<?, ?it/s]

GNN accuracy: 0.898500 vs MWPM: 0.982810

Trying train_size = 10,000,000
  Chunk 1/1: 10,000,000 samples


  0%|          | 0/39062 [00:00<?, ?it/s]

GNN accuracy: 0.900500 vs MWPM: 0.982810

Trying train_size = 100,000,000
  Chunk 1/10: 10,000,000 samples


  0%|          | 0/39062 [00:00<?, ?it/s]

In [17]:
# ============================================================================
# QUICK TEST - Run this first to verify everything works
# ============================================================================

print("🧪 Quick Test: Training a small GNN on d=3")
print("="*50)

# Create trainer
test_trainer = GNNTrainer(p=0.005, d=3, device=device)

# Generate small dataset
print("\nGenerating 10,000 training samples...")
syndromes, labels = test_trainer.sample_data(10000)
print(f"✓ Syndromes shape: {syndromes.shape}")
print(f"✓ Labels shape: {labels.shape}")
print(f"✓ Example syndrome: {syndromes[0, :5].tolist()} ...")

# Train for 3 epochs
print("\nTraining for 3 epochs...")
for epoch in range(3):
    loss, acc = test_trainer.train_epoch(syndromes, labels, batch_size=256)
    print(f"  Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

# Evaluate
print("\nEvaluating on fresh test data...")
test_acc = test_trainer.evaluate(5000)
print(f"✓ Test Accuracy: {test_acc:.4f}")

print("\n" + "="*50)
print("✅ Quick test passed! The GNN is working.")
print("="*50)

🧪 Quick Test: Training a small GNN on d=3

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 56 connections
  Average neighbors per detector: 2.3

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


Generating 10,000 training samples...


RuntimeError: Could not infer dtype of numpy.int64